# 01 - Data Preprocessing

This notebook loads the original CICIDS2017 dataset and performs one preprocessing task at a time. It creates one processed file for model training.

## Step 1 - Import the libraries

In [1]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler

## Step 2 - Set the file paths

In [2]:
PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_FILE = PROJECT / 'data' / 'CICIDS2017_combined.csv'
PROCESSED_FILE = PROJECT / 'data' / 'processed_data.npz'
MODELS_DIR = PROJECT / 'models'
RESULTS_DIR = PROJECT / 'results' / 'preprocessing'
MODELS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42
TEST_SIZE = 0.20
VALIDATION_SIZE = 0.10

## Step 3 - Load the original dataset

Keep `ROWS_TO_LOAD = None` for the complete assignment dataset. The complete CSV is large, so close unnecessary applications before running this cell.

In [3]:
ROWS_TO_LOAD = None

if not DATA_FILE.exists():
    raise FileNotFoundError(f'Dataset not found: {DATA_FILE}')

df = pd.read_csv(DATA_FILE, nrows=ROWS_TO_LOAD, low_memory=False)
original_rows = len(df)
print('Dataset shape:', df.shape)
display(df.head())

Dataset shape: (2830743, 80)


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,_source_file
0,49188,4,2,0,12,0,6,6,6.0,0.0,...,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,Monday-WorkingHours.pcap_ISCX.csv
1,49188,1,2,0,12,0,6,6,6.0,0.0,...,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,Monday-WorkingHours.pcap_ISCX.csv
2,49188,1,2,0,12,0,6,6,6.0,0.0,...,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,Monday-WorkingHours.pcap_ISCX.csv
3,49188,1,2,0,12,0,6,6,6.0,0.0,...,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,Monday-WorkingHours.pcap_ISCX.csv
4,49486,3,2,0,12,0,6,6,6.0,0.0,...,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,Monday-WorkingHours.pcap_ISCX.csv


## Step 4 - Clean the column names

In [4]:
df.columns = [str(column).strip() for column in df.columns]
print('Column names cleaned.')

Column names cleaned.


## Step 5 - Remove duplicated columns

In [5]:
duplicate_columns = [
    column for column in df.columns
    if column.endswith('.1') and column[:-2] in df.columns
]
df = df.drop(columns=duplicate_columns)
print('Duplicated columns removed:', duplicate_columns)

Duplicated columns removed: ['Fwd Header Length.1']


## Step 6 - Remove duplicated rows

In [6]:
columns_used_for_duplicates = [column for column in df.columns if column != '_source_file']
rows_before_duplicates = len(df)
df = df.drop_duplicates(subset=columns_used_for_duplicates).reset_index(drop=True)
duplicates_removed = rows_before_duplicates - len(df)
print('Duplicated rows removed:', duplicates_removed)

Duplicated rows removed: 308381


## Step 7 - Clean and group the labels

In [7]:
df['Label'] = df['Label'].astype(str).str.strip()

def group_label(label):
    label = label.replace('–', '-').replace('�', '-')
    if label == 'BENIGN':
        return 'BENIGN'
    if label == 'Bot':
        return 'Botnet'
    if label == 'DDoS':
        return 'DDoS'
    if label.startswith('DoS'):
        return 'DoS'
    if label.startswith('Web Attack'):
        return 'Web Attack'
    if 'Patator' in label:
        return 'Brute Force'
    if label in ['PortScan', 'Infiltration', 'Heartbleed']:
        return label
    return None

df['attack_type'] = df['Label'].map(group_label)
unknown_labels = df.loc[df['attack_type'].isna(), 'Label'].unique()
if len(unknown_labels):
    raise ValueError(f'Unknown labels: {unknown_labels}')

display(df['attack_type'].value_counts().to_frame('rows'))

,rows
attack_type,
BENIGN,2096484
DoS,193748
DDoS,128016
PortScan,90819
Brute Force,9152
Web Attack,2143
Botnet,1953
Infiltration,36
Heartbleed,11


## Step 8 - Separate the features and target

In [8]:
columns_not_used_for_training = ['Label', 'attack_type', '_source_file']
feature_columns = [
    column for column in df.columns
    if column not in columns_not_used_for_training
]
X = df[feature_columns].copy()
y = df['attack_type'].copy()
print('Feature shape:', X.shape)
print('Target shape:', y.shape)

Feature shape: (2522362, 77)
Target shape: (2522362,)


## Step 9 - Convert the features to numeric values

In [9]:
X = X.apply(pd.to_numeric, errors='coerce')
print('All features converted to numeric values.')

All features converted to numeric values.


## Step 10 - Replace infinity values

In [10]:
X = X.replace([np.inf, -np.inf], np.nan)
empty_columns = X.columns[X.isna().all()].tolist()
X = X.drop(columns=empty_columns)
feature_columns = X.columns.tolist()
print('Missing values to fill:', int(X.isna().sum().sum()))

Missing values to fill: 3128


## Step 11 - Split the data into training, validation and test sets

In [11]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=VALIDATION_SIZE / (1 - TEST_SIZE),
    random_state=RANDOM_STATE,
    stratify=y_train_full,
)

print('Training rows:', len(X_train))
print('Validation rows:', len(X_val))
print('Test rows:', len(X_test))

Training rows: 1765652
Validation rows: 252237
Test rows: 504473


## Step 12 - Create the feature preprocessor

In [12]:
preprocessor = Pipeline([
    ('fill_missing_values', SimpleImputer(strategy='median')),
    ('remove_constant_features', VarianceThreshold()),
    ('standardize_features', StandardScaler()),
])
preprocessor

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('fill_missing_values', ...), ('remove_constant_features', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows 

## Step 13 - Fit the preprocessor using only the training data

In [13]:
X_train_processed = preprocessor.fit_transform(X_train).astype(np.float32)
print('Processed training shape:', X_train_processed.shape)

Processed training shape: (1765652, 69)


## Step 14 - Apply the same preprocessing to the validation and test data

In [14]:
X_val_processed = preprocessor.transform(X_val).astype(np.float32)
X_test_processed = preprocessor.transform(X_test).astype(np.float32)
print('Processed validation shape:', X_val_processed.shape)
print('Processed test shape:', X_test_processed.shape)

Processed validation shape: (252237, 69)
Processed test shape: (504473, 69)


## Step 15 - Encode the class labels

In [15]:
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train).astype(np.int64)
y_val_encoded = label_encoder.transform(y_val).astype(np.int64)
y_test_encoded = label_encoder.transform(y_test).astype(np.int64)
print('Encoded classes:', list(label_encoder.classes_))

Encoded classes: ['BENIGN', 'Botnet', 'Brute Force', 'DDoS', 'DoS', 'Heartbleed', 'Infiltration', 'PortScan', 'Web Attack']


## Step 16 - Save one processed dataset

In [16]:
np.savez_compressed(
    PROCESSED_FILE,
    X_train=X_train_processed,
    X_val=X_val_processed,
    X_test=X_test_processed,
    y_train=y_train_encoded,
    y_val=y_val_encoded,
    y_test=y_test_encoded,
    train_index=X_train.index.to_numpy(),
    val_index=X_val.index.to_numpy(),
    test_index=X_test.index.to_numpy(),
    feature_names=np.array(preprocessor.get_feature_names_out(feature_columns)),
)
print('Saved:', PROCESSED_FILE)

Saved: C:\Users\Matheesha\Desktop\4th year\Y4 S1\Deep Learning\Assignment\SE4050-DL\data\processed_data.npz


## Step 17 - Save the preprocessing objects

In [17]:
joblib.dump(preprocessor, MODELS_DIR / 'preprocessor.joblib')
joblib.dump(label_encoder, MODELS_DIR / 'label_encoder.joblib')
print('Preprocessing objects saved in the models folder.')

Preprocessing objects saved in the models folder.


## Step 18 - Check the final output and confirm there is no leakage between the splits

In [18]:
assert np.isfinite(X_train_processed).all()
assert np.isfinite(X_val_processed).all()
assert np.isfinite(X_test_processed).all()
assert len(X_train_processed) == len(y_train_encoded)
assert len(X_val_processed) == len(y_val_encoded)
assert len(X_test_processed) == len(y_test_encoded)

train_rows, val_rows, test_rows = set(X_train.index), set(X_val.index), set(X_test.index)
assert not train_rows & val_rows, 'Training and validation rows overlap'
assert not train_rows & test_rows, 'Training and test rows overlap'
assert not val_rows & test_rows, 'Validation and test rows overlap'

print('Original rows:', original_rows)
print('Cleaned rows:', len(df))
print('Duplicates removed:', duplicates_removed)
print('Number of features:', X_train_processed.shape[1])
print('No row appears in more than one split.')
print('Preprocessing completed successfully.')

Original rows: 2830743
Cleaned rows: 2522362
Duplicates removed: 308381
Number of features: 69
No row appears in more than one split.
Preprocessing completed successfully.


## Step 19 - Save the class distribution of each split

In [19]:
split_distribution = pd.DataFrame({
    'train': y_train.value_counts(),
    'validation': y_val.value_counts(),
    'test': y_test.value_counts(),
}).fillna(0).astype(int).sort_values('train', ascending=False)
split_distribution['train_%'] = (100 * split_distribution['train'] / split_distribution['train'].sum()).round(4)
split_distribution.to_csv(RESULTS_DIR / 'class_distribution_by_split.csv')
display(split_distribution)

,train,validation,test,train_%
attack_type,,,,
BENIGN,1467538,209649,419297,83.1159
DoS,135623,19375,38750,7.6812
DDoS,89611,12802,25603,5.0752
PortScan,63573,9082,18164,3.6005
Brute Force,6407,915,1830,0.3629
Web Attack,1500,214,429,0.0850
Botnet,1367,195,391,0.0774
Infiltration,25,4,7,0.0014
Heartbleed,8,1,2,0.0005
